# CrewAI — Complete Agentic AI & Multi-Agent Masterclass (2026)

**Build an end-to-end Agentic AI system with CrewAI Crews, Flows, tools, memory, state, checkpoints, approvals, knowledge, observability, and evaluation.**

- **Audience:** advanced Python, GenAI, LangGraph and Agentic AI students.
- **Reference release:** CrewAI **1.15.26**, released 8 Oct 2026; APIs checked against versioned CrewAI 1.15.x official docs (some web docs snapshot at 1.15.14). Never assume a future release stays identical.
- **Execution policy:** free offline teaching examples work without any API key; external model calls are **OFF by default**. Marked optional blocks may incur usage charges.
- **Python:** 3.10–3.13 inclusive (current package metadata is `>=3.10,<3.14`). Python 3.11 or 3.12 recommended for workshops.
- **Demo project:** **Multi-Agent IT Incident Resolution System**, using **mock/simulated** logs, metrics, deployment information and remediation; it never changes a real production system.
- **Accuracy:** this is a teaching notebook, not a promise that all optional cloud integrations work without configuration. It includes validation exercises and official URLs in the last section.

### Notebook operating modes

| Flag | Default | What it does |
|---|---|---|
| `RUN_LIVE` | `False` | Executes live CrewAI + LLM agent tasks; needs API key + dependencies |
| `RUN_FLOW_DEMOS` | `False` | Executes local CrewAI Flow examples (no LLM required) |
| `RUN_MEMORY_DEMOS` | `False` | Runs CrewAI memory (model/embedding calls likely) |
| `RUN_INTERACTIVE` | `False` | Enables interactive approval prompts |
| `RUN_OPTIONAL` | `False` | Enables optional integrations needing extra setup |

**Important security note:** treat tool output and retrieved documents as untrusted input. Do not place secrets inside notebook cells or give autonomous agents unrestricted write tools.

## 0. Your LangGraph classes → CrewAI concept map


This notebook maps all major topics from your **LangGraph classes 41–50** into CrewAI. CrewAI is an **independent** framework; its abstractions are not always one-to-one with LangGraph.

| LangGraph topic | CrewAI method / feature | Key difference |
|---|---|---|
| Introduction/StateGraph | Crews + Flows | A Flow is event-driven; it is not literally `StateGraph` |
| ReAct + tool calling | `Agent`, `Task`, custom `@tool`, tool loop | Crew agent is role-oriented |
| Pydantic | `Task(output_pydantic=...)`, Flow `BaseModel` state | Task outputs and Flow state are different concepts |
| Memory + LangMem | `Memory`, scopes, Crew memory, external storage | Unified Memory introduced; embedding/LLM backend needed |
| Checkpointing | `CheckpointConfig`, `@persist` | Checkpoints differ from Flow state persistence |
| Human approval | `Task(human_input=True)`, `@human_feedback` | In-class input is not enterprise approval queue |
| Conditional edges | `@router` with labeled `@listen` | Event routing rather than StateGraph edge object |
| Subgraphs | Nested Flows, Flow→Crew, multiple Crews | Workflows compose using Python |
| Parallel orchestration | asynchronous tasks/Flow branches | Use explicit dependencies for fan-in |
| Command control flow | `@router`, Python decisions, crews/process | No identical LangGraph `Command` object |
| Multi-agent | sequential/hierarchical Crews, Flows+Crews | Specialized roles and manager mechanisms |

In [ ]:
langgraph_mapping = {
    'tool_calling': 'Agent + custom tools',
    'state': 'Flow state (Pydantic/dict)',
    'conditional_edges': '@router + @listen',
    'checkpoint': 'CheckpointConfig (execution) or @persist (Flow state)',
    'multi_agent': 'Crew(Process.sequential/hierarchical) + Flow orchestration',
    'human_in_loop': 'Task(human_input=True) or @human_feedback',
}
for k, v in langgraph_mapping.items():
    print(f'{k:22s} -> {v}')

## 1. Environment installation and safe run switches


Install in a fresh Python environment. `%pip` may be used inside Jupyter manually. The cell below only **prints** the install command by default so `Run All` does not change the environment. `crewai-tools` is optional but used in the advanced sections. Use `uv pip install` instead of pip if working in a `uv` virtual environment.

Official installation: https://docs.crewai.com/en/installation

In [ ]:
import os, sys, json, time, sqlite3, asyncio, importlib.util
from pathlib import Path
from datetime import datetime, timezone
from typing import Optional, Literal
from pydantic import BaseModel, Field, ValidationError

RUN_LIVE = False
RUN_FLOW_DEMOS = False
RUN_MEMORY_DEMOS = False
RUN_INTERACTIVE = False
RUN_OPTIONAL = False
MODEL = os.getenv('CREWAI_DEMO_MODEL', 'openai/gpt-4o-mini')
WORKDIR = Path('crewai_masterclass_outputs')
WORKDIR.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version.split()[0])
print('CrewAI installed:', importlib.util.find_spec('crewai') is not None)
print('To install: python -m pip install "crewai==1.15.26" "crewai-tools==1.15.26" python-dotenv')

## 2. Secrets, LLM provider selection, model and notebook helpers


For OpenAI-based examples set `OPENAI_API_KEY` in the environment (or `.env` loaded with `python-dotenv`). Never hardcode real secrets or commit `.env`. CrewAI also supports other providers; availability, credentials and model names depend on the provider.

In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

def check_live_ready():
    if importlib.util.find_spec('crewai') is None:
        raise RuntimeError('Install CrewAI first.')
    if not os.getenv('OPENAI_API_KEY') and MODEL.startswith('openai/'):
        raise RuntimeError('Set OPENAI_API_KEY in your environment first.')

if RUN_LIVE:
    check_live_ready()
    from crewai import Agent, Task, Crew, Process, LLM
    demo_llm = LLM(model=MODEL, temperature=0)
else:
    print('Live model calls disabled. Turn RUN_LIVE=True only after setup.')

## 3. First agent + Task + Crew (the minimum useful program)


A CrewAI `Agent` has a **role, goal, backstory**. A `Task` defines an assignment and `expected_output`. A `Crew` groups agents and tasks; `Process.sequential` executes in task order.

In [ ]:
if RUN_LIVE:
    researcher = Agent(role='Documentation Researcher',
        goal='Explain the topic accurately', backstory='Careful technical educator',
        llm=demo_llm, verbose=False)
    starter_task = Task(description='Explain 3 key benefits of agent tools.',
        expected_output='Three concise numbered benefits.', agent=researcher)
    starter_crew = Crew(agents=[researcher], tasks=[starter_task],
        process=Process.sequential, verbose=False)
    starter_result = starter_crew.kickoff()
    print(starter_result.raw)

## 4. ReAct-style reasoning and automatic tool use


CrewAI coordinates agent decisions and tool execution. The exact private reasoning is not an API contract. Observe **tool calls and outputs**, not chain-of-thought. Keep `max_iter` bounded.

In [ ]:
if RUN_LIVE:
    from crewai.tools import tool
    @tool('Multiply numbers')
    def multiply_numbers(x: float, y: float) -> str:
        """Multiply two numeric values precisely."""
        return str(x * y)
    calculator_agent = Agent(role='Math Solver', goal='Compute arithmetic with tools',
        backstory='Always use calculator tools for numerical work',
        tools=[multiply_numbers], llm=demo_llm, max_iter=5)
    math_task = Task(description='What is 23 multiplied by 17? Use the tool.',
        expected_output='The correct number with short explanation.', agent=calculator_agent)
    print(Crew(agents=[calculator_agent], tasks=[math_task]).kickoff().raw)

## 5. Two ways to build tools: `@tool` and `BaseTool`


- Use `@tool` for simple typed functions.
- Use `BaseTool` when you need custom class state, input schema and reusable behaviors.
- Treat tool arguments as untrusted and whitelist allowed resources.

In [ ]:
if RUN_LIVE:
    from typing import Type
    from crewai.tools import tool, BaseTool
    @tool('Safe Ticket Lookup')
    def safe_ticket_lookup(ticket_id: str) -> str:
        """Look up one demo incident by ticket identifier."""
        if ticket_id != 'INC-1001':
            return 'Unknown incident'
        return 'INC-1001: checkout API latency increased after release.'

    class TicketArgs(BaseModel):
        ticket_id: str = Field(description='Example INC-1001')

    class TicketLookupTool(BaseTool):
        name: str = 'ticket_lookup'
        description: str = 'Read a simulated IT ticket by ID'
        args_schema: Type[BaseModel] = TicketArgs
        def _run(self, ticket_id: str) -> str:
            return 'checkout latency alert' if ticket_id == 'INC-1001' else 'not found'

    print(TicketLookupTool().run(ticket_id='INC-1001'))

## 6. Pydantic tool schemas and validation (offline exercise)


Pydantic validation catches invalid tool input before business logic. This is independent of an LLM and can be executed safely offline.

In [ ]:
class IncidentQuery(BaseModel):
    incident_id: str = Field(pattern=r'^INC-[0-9]{4}$')
    limit: int = Field(default=10, ge=1, le=100)

print(IncidentQuery(incident_id='INC-1001', limit=5).model_dump())
try:
    IncidentQuery(incident_id='../secrets', limit=500)
except ValidationError as e:
    print('Invalid inputs correctly blocked:', len(e.errors()), 'errors')

## 7. Structured task output with `output_pydantic` and `output_json`


A `Task(output_pydantic=YourModel)` asks CrewAI to validate and expose `result.pydantic`. `output_json` is another option. `CrewOutput.raw` is plain text. **Do not assume** a Pydantic instance is directly the return value of `kickoff`.

In [ ]:
class IncidentAssessment(BaseModel):
    incident_id: str
    severity: Literal['low','medium','high','critical']
    summary: str
    probable_cause: str

print(IncidentAssessment(incident_id='INC-1001', severity='high',
    summary='Elevated checkout latency', probable_cause='Unverified deployment correlation').model_dump())
if RUN_LIVE:
    analysis_agent = Agent(role='Incident Analyst', goal='Produce factual incident assessments',
        backstory='Careful SRE specialist', llm=demo_llm)
    assess_task = Task(description='Analyze INC-1001: checkout latency after deploy; do not claim certainty.',
        expected_output='Structured severity, summary and possible cause.',
        agent=analysis_agent, output_pydantic=IncidentAssessment)
    r = Crew(agents=[analysis_agent], tasks=[assess_task]).kickoff()
    print(r.pydantic, r.raw)

## 8. Multi-agent sequential Crew: research → analysis → report


Sequential crews pass context from completed tasks to subsequent tasks. Use `context=[task]` explicitly when task dependencies must be clear.

In [ ]:
if RUN_LIVE:
    collector = Agent(role='Evidence Collector', goal='Summarize given evidence',
        backstory='SRE investigation engineer', llm=demo_llm)
    reporter = Agent(role='Incident Reporter', goal='Write careful postmortems',
        backstory='Experienced technical writer', llm=demo_llm)
    evidence_task = Task(description='Extract relevant evidence from: checkout 5xx rate 8%, deployment at 10:02.',
        expected_output='Two factual evidence bullets.', agent=collector)
    report_task = Task(description='Write a short incident summary and clearly label hypotheses.',
        expected_output='Brief report with evidence vs hypotheses.', agent=reporter,
        context=[evidence_task])
    sequential = Crew(agents=[collector, reporter], tasks=[evidence_task, report_task],
        process=Process.sequential)
    print(sequential.kickoff().raw)

## 9. Hierarchical Crew with a manager (delegation)


In `Process.hierarchical`, a manager allocates work to specialist agents. Configure `manager_llm` or `manager_agent`. A hierarchical crew is **not** deterministic: the manager chooses allocation.

In [ ]:
if RUN_LIVE:
    logs_agent = Agent(role='Logs Specialist', goal='Analyze known log evidence',
        backstory='Expert in distributed app logs', llm=demo_llm)
    metrics_agent = Agent(role='Metrics Specialist', goal='Analyze monitoring anomalies',
        backstory='SRE familiar with p95 and error rate', llm=demo_llm)
    manager_task = Task(description='Coordinate analysis of 5xx errors and checkout p95 regression.',
        expected_output='Evidence-backed investigation and recommended next steps.')
    hierarchical = Crew(agents=[logs_agent, metrics_agent], tasks=[manager_task],
        process=Process.hierarchical, manager_llm=demo_llm,
        max_rpm=15, verbose=False)
    print(hierarchical.kickoff().raw)

## 10. Agent delegation vs fixed role assignment


`allow_delegation=True` permits agent task collaboration when a crew context makes teammates available. In a sequential process, direct task ownership is more predictable; hierarchy lets a manager decide which worker handles each task.

In [ ]:
if RUN_LIVE:
    lead = Agent(role='Lead Investigator', goal='Get grounded answers from specialists',
        backstory='You assign specialist work rather than guessing', llm=demo_llm,
        allow_delegation=True, max_iter=8)
    specialist = Agent(role='Database Specialist', goal='Review database latency evidence',
        backstory='Database performance engineer', llm=demo_llm)
    print('Delegation enabled:', lead.allow_delegation)
    # In a suitable Crew, lead may be able to delegate to colleagues.

## 11. Task context, async_execution and explicit fan-in


A common parallel pattern: start independent tasks with `async_execution=True`, then a downstream task uses `context=[task_a, task_b]`. Avoid data races or shared mutable state.

In [ ]:
if RUN_LIVE:
    a = Agent(role='Logs Analyst', goal='Find log anomalies', backstory='SRE logs', llm=demo_llm)
    b = Agent(role='Metrics Analyst', goal='Find metrics anomalies', backstory='SRE metrics', llm=demo_llm)
    c = Agent(role='Synthesizer', goal='Combine evidence carefully', backstory='Incident commander', llm=demo_llm)
    t1 = Task(description='Read sample logs: HTTP 502 8%, timeout increasing.',
        expected_output='Log findings.', agent=a, async_execution=True)
    t2 = Task(description='Read sample metrics: p95 2300ms, baseline 200ms.',
        expected_output='Metric findings.', agent=b, async_execution=True)
    t3 = Task(description='Synthesize the first two findings, labeling uncertainty.',
        expected_output='One short synthesis.', agent=c, context=[t1,t2])
    print(Crew(agents=[a,b,c], tasks=[t1,t2,t3], process=Process.sequential).kickoff().raw)

## 12. Task callbacks and data exchange


`Task(callback=...)` fires on completion. Keep callbacks idempotent and avoid writing sensitive data to logs.

In [ ]:
callback_events = []
def audit_task_output(output):
    callback_events.append({'time': datetime.now(timezone.utc).isoformat(),
                            'preview': str(getattr(output, 'raw', output))[:100]})

if RUN_LIVE:
    callback_agent = Agent(role='Summary Writer', goal='Produce summaries',
        backstory='Concise technical writer', llm=demo_llm)
    callback_task = Task(description='Write 2 lines about retry budgets.',
        expected_output='2 lines.', agent=callback_agent, callback=audit_task_output)
    Crew(agents=[callback_agent], tasks=[callback_task]).kickoff()
    print(callback_events)

## 13. Guardrails: deterministic task output validation


CrewAI task guardrails can return `(True, value)` or `(False, feedback)`. Use `guardrail_max_retries` to bound regeneration. Guardrails are not a substitute for application-level authorization.

In [ ]:
def minimum_evidence_guardrail(task_output):
    txt = str(getattr(task_output, 'raw', task_output))
    if 'evidence' not in txt.lower():
        return (False, "Include an 'Evidence:' section with facts.")
    return (True, txt)

from types import SimpleNamespace
assert minimum_evidence_guardrail(SimpleNamespace(raw='Evidence: log rate=8%'))[0]
assert not minimum_evidence_guardrail(SimpleNamespace(raw='Probably a memory leak'))[0]
print('Offline task-guardrail contract passes')
if RUN_LIVE:
    guard_agent = Agent(role='Incident Writer', goal='Ground all assertions',
        backstory='SRE writer', llm=demo_llm)
    guarded = Task(description='Summarize checkout 5xx rate 8% using an Evidence section.',
        expected_output='Summary with Evidence section.', agent=guard_agent,
        guardrail=minimum_evidence_guardrail, guardrail_max_retries=2)
    print(Crew(agents=[guard_agent], tasks=[guarded]).kickoff().raw)

## 14. LLM-based guardrails and agent reasoning settings


Newer CrewAI supports string-style task guardrails and `reasoning=True` for agent planning. These are extra model invocations; benchmark cost and guardrail quality before enabling globally.

In [ ]:
if RUN_LIVE:
    reasoning_agent = Agent(role='Reliability Planner', goal='Plan low-risk diagnosis',
        backstory='Platform engineer', llm=demo_llm, reasoning=True,
        max_reasoning_attempts=2, max_iter=10)
    reasoning_task = Task(description='Plan how to investigate slow requests without changing production.',
        expected_output='A reversible, read-only numbered plan.', agent=reasoning_agent,
        guardrail='The answer must avoid recommending destructive commands as immediate actions.')
    print(Crew(agents=[reasoning_agent], tasks=[reasoning_task]).kickoff().raw)

## 15. Crew planning, cache and cost-control settings


`planning=True` creates a planning step for a Crew; control `planning_llm` explicitly for reproducible costs. `max_rpm` and bounded `max_iter` help reduce runaway calls. Tool caching is separate from application response caching.

In [ ]:
if RUN_LIVE:
    p_agent = Agent(role='Project Planner', goal='Sequence investigation tasks',
        backstory='Incident commander', llm=demo_llm, max_iter=6, cache=True)
    p_task = Task(description='Plan three read-only checks for an HTTP 502 incident.',
        expected_output='Three checks in order.', agent=p_agent)
    planned = Crew(agents=[p_agent], tasks=[p_task], process=Process.sequential,
        planning=True, planning_llm=demo_llm, max_rpm=10, cache=True)
    print(planned.kickoff().raw)

## 16. Flow introduction: `Flow`, `@start`, `@listen`


Use Flows when exact event order matters. Below is an **offline** Flow: no LLM is called, but CrewAI must be installed to execute it. `@listen` receives the previous method's return value.

In [ ]:
if RUN_FLOW_DEMOS:
    from crewai.flow.flow import Flow, start, listen
    class HelloFlow(Flow):
        @start()
        def begin(self):
            return 'INC-1001'
        @listen(begin)
        def summarize(self, incident_id):
            return f'Initialized investigation for {incident_id}'
    hello_flow = HelloFlow()
    print(hello_flow.kickoff())

## 17. Strongly typed Flow state using Pydantic


Flow state can be a Pydantic model. The state has a unique `id`; fields persist within the Flow and can be referenced by listeners. This is conceptually similar to sharing application state, but not to LangGraph's reducer semantics.

In [ ]:
class InvestigationState(BaseModel):
    incident_id: str = 'INC-1001'
    phase: str = 'new'
    evidence: list[str] = Field(default_factory=list)

if RUN_FLOW_DEMOS:
    from crewai.flow.flow import Flow, start, listen
    class TypedIncidentFlow(Flow[InvestigationState]):
        @start()
        def collect(self):
            self.state.phase = 'triage'
            self.state.evidence.append('checkout p95=2300ms')
            return 'collected'
        @listen(collect)
        def report(self, _):
            return self.state.model_dump()
    typed_flow = TypedIncidentFlow()
    print(typed_flow.kickoff())

## 18. Flow routing = LangGraph conditional edge equivalent


Use `@router(method)` to return a label. Then define `@listen('route_label')` methods. Only the matching route should be selected. All routing conditions should be deterministic and testable.

In [ ]:
if RUN_FLOW_DEMOS:
    from crewai.flow.flow import Flow, start, router, listen
    class SeverityFlow(Flow):
        @start()
        def triage(self):
            self.state['error_rate_pct'] = 8.0
            return self.state['error_rate_pct']
        @router(triage)
        def select_path(self, rate):
            return 'urgent' if rate >= 5 else 'normal'
        @listen('urgent')
        def urgent(self):
            return 'Escalate to human incident commander'
        @listen('normal')
        def normal(self):
            return 'Collect routine diagnostics'
    print(SeverityFlow().kickoff())

## 19. Flow `and_` and `or_`: fan-in and alternative triggers


`and_` waits until both named events have completed. `or_` fires when either completes, and might execute more than once. In event-driven designs this is important for idempotency and duplication avoidance.

In [ ]:
if RUN_FLOW_DEMOS:
    from crewai.flow.flow import Flow, start, listen, and_, or_
    class JoinFlow(Flow):
        @start()
        def collect_logs(self):
            self.state['logs']='5xx elevated'
            return 'logs-ready'
        @start()
        def collect_metrics(self):
            self.state['metrics']='p95 high'
            return 'metrics-ready'
        @listen(and_(collect_logs,collect_metrics))
        def join(self):
            return {'logs':self.state['logs'], 'metrics':self.state['metrics']}
    print(JoinFlow().kickoff())

## 20. Parallel Flow execution and asyncio fan-out / fan-in


Flow async listeners can perform concurrent read-only operations. For application-level control, `asyncio.gather` gives explicit fan-out/fan-in and exception handling. Below uses only standard Python.

In [ ]:
async def fake_fetch(name, delay=0.01):
    await asyncio.sleep(delay)
    return {'name': name, 'status': 'ok'}
async def collect_parallel():
    return await asyncio.gather(fake_fetch('logs'),fake_fetch('metrics'),fake_fetch('deployments'))
print(asyncio.run(collect_parallel()) if 'ipykernel' not in sys.modules else 'In Jupyter: await collect_parallel()')

## 21. Bounded evaluator–optimizer loops


Never let an agent continuously revise forever. Use a maximum attempt budget, success criteria and logged reviewer feedback. This is an **offline orchestration pattern** that can be inserted into a Flow.

In [ ]:
def bounded_refinement(draft: str, min_words=5, max_attempts=3):
    history = []
    for attempt in range(1, max_attempts+1):
        passing = len(draft.split()) >= min_words
        history.append({'attempt': attempt, 'passes': passing})
        if passing:
            return draft, history
        draft += ' Include evidence and a clear uncertainty statement.'
    return draft, history

print(bounded_refinement('Elevated 5xx'))

## 22. Subgraph-like composition: Flow calling a Crew


A Flow method can invoke a specialized Crew, return its result and pass that output to another listener. This is the canonical hybrid arrangement: deterministic process outside; autonomous agents inside.

In [ ]:
if RUN_LIVE:
    from crewai.flow.flow import Flow, start, listen
    class HybridFlow(Flow):
        @start()
        def run_specialist_crew(self):
            agent = Agent(role='Log Expert', goal='Extract evidence', backstory='SRE', llm=demo_llm)
            task = Task(description='Summarize: INC-1001 log error=HTTP 502.',
                expected_output='A short log finding.', agent=agent)
            return Crew(agents=[agent], tasks=[task]).kickoff().raw
        @listen(run_specialist_crew)
        def finalize(self, evidence):
            return {'evidence':evidence, 'needs_human_review':True}
    print(HybridFlow().kickoff())

## 23. Reusing child Flows (nested orchestration)


A Flow may invoke a smaller Flow; the caller controls what data moves across the boundary. Think of it as composing Python components rather than exact LangGraph subgraph messages.

In [ ]:
if RUN_FLOW_DEMOS:
    from crewai.flow.flow import Flow, start, listen
    class DataCollectionFlow(Flow):
        @start()
        def run(self):
            return {'logs': ['502 upstream error'], 'metrics':['p95=2300ms']}
    class MainFlow(Flow):
        @start()
        def acquire(self):
            return DataCollectionFlow().kickoff()
        @listen(acquire)
        def evaluate(self, data):
            return {'source_count':len(data), 'keys':list(data)}
    print(MainFlow().kickoff())

## 24. Flow state persistence via `@persist` (SQLite-backed)


`@persist` saves state across Flow methods; using the same state ID can reload data. Don't mix up persist-state hydration and `CheckpointConfig` runtime checkpoint restore: they solve related but distinct problems.

In [ ]:
if RUN_FLOW_DEMOS:
    from crewai.flow.flow import Flow, start
    from crewai.flow.persistence import persist
    class CounterState(BaseModel):
        id: str = ''
        counter: int = 0
    @persist
    class CounterFlow(Flow[CounterState]):
        @start()
        def count(self):
            self.state.counter += 1
            return self.state.counter
    cf = CounterFlow()
    print('first:', cf.kickoff(), 'id:', cf.state.id)
    replay_cf = CounterFlow()
    print('same ID next run:', replay_cf.kickoff(inputs={'id': cf.state.id}))

## 25. Fork a persisted Flow into a new history


The `restore_from_state_id=` option loads a stored state into a fresh flow ID. That preserves the old history rather than overwriting it. Use versioned state schema migration in long-running services.

In [ ]:
if RUN_FLOW_DEMOS:
    branch = CounterFlow()  # defined in prior persistence cell
    branch.kickoff(restore_from_state_id=cf.state.id)
    assert branch.state.id != cf.state.id
    print('source:', cf.state.id, 'fork:', branch.state.id, 'fork counter:', branch.state.counter)

## 26. Crew/Agent/Flow execution checkpointing via CheckpointConfig


In CrewAI 1.15.x checkpointing is **native**. `CheckpointConfig` can capture task completion state, restore progress and fork a run. JsonProvider and SqliteProvider exist. Checkpoint files can contain sensitive data: restrict permissions, encrypt and avoid unserializing untrusted snapshots.

In [ ]:
if RUN_LIVE:
    from crewai import CheckpointConfig
    from crewai.state import SqliteProvider
    cp_agent = Agent(role='Investigator', goal='Read simulated evidence',
        backstory='SRE', llm=demo_llm)
    cp_task = Task(description='Summarize p95=2300ms, errors=8%.',
        expected_output='Evidence summary.', agent=cp_agent)
    checkpointed = Crew(agents=[cp_agent], tasks=[cp_task],
        checkpoint=CheckpointConfig(location=str(WORKDIR/'crewai_checkpoints.db'),
            provider=SqliteProvider(), max_checkpoints=10))
    print(checkpointed.kickoff().raw)
    # To resume: checkpointed.kickoff(from_checkpoint=CheckpointConfig(restore_from='TRUSTED_PATH'))

## 27. Memory architecture: unified Memory vs old short/long/entity types


Current CrewAI docs describe one `Memory` class with semantic/recency/importance recall, hierarchical scopes, consolidation and crew integration. **Memory may require both model calls and embedding calls**. Do not assume enabling memory is a free SQLite-only operation.

In [ ]:
if RUN_MEMORY_DEMOS:
    check_live_ready()
    from crewai import Memory
    crew_memory = Memory(recency_weight=0.3, semantic_weight=0.5, importance_weight=0.2)
    crew_memory.remember('Checkout service deployment changed upstream timeout on 2026-10-10.',
                         scope='/incidents/INC-1001')
    matches = crew_memory.recall('What changed in checkout?', scope='/incidents/INC-1001', limit=3)
    for match in matches:
        print(match.score, match.record.content)
    crew_memory.close()

## 28. Scoped private memory and per-tenant isolation


Use scopes such as `/tenant/<id>/incident/<id>`. Scope views narrow retrieval; application IAM and tenant authorization must still be enforced separately.

In [ ]:
if RUN_MEMORY_DEMOS:
    from crewai import Memory
    memory = Memory()
    alice_view = memory.scope('/tenant/alice')
    bob_view = memory.scope('/tenant/bob')
    alice_view.remember('Alice project uses PostgreSQL.')
    print([m.record.content for m in alice_view.recall('database', limit=3)])
    print([m.record.content for m in bob_view.recall('database', limit=3)])
    memory.close()

## 29. Crew memory enabled and memory-aware agent behavior


`Crew(memory=True)` makes CrewAI retrieve relevant context and store extracted findings. Use dedicated embedding config when avoiding OpenAI embeddings; default memory embeddings may use `text-embedding-3-large`.

In [ ]:
if RUN_LIVE and RUN_MEMORY_DEMOS:
    memory_agent = Agent(role='Incident Historian', goal='Recall relevant past incidents',
        backstory='Experienced reliability engineer', llm=demo_llm)
    memory_task = Task(description='Explain similarities between checkout failures and upstream timeouts.',
        expected_output='Concise similarities with uncertainty.', agent=memory_agent)
    remembered = Crew(agents=[memory_agent], tasks=[memory_task],
                      memory=True, process=Process.sequential)
    print(remembered.kickoff().raw)

## 30. Offline long-term memory pattern with SQLite


This is **not** CrewAI `Memory`; it is an explicit application-owned SQL store for rules, owner notes and human-approved facts. Keyword retrieval here is just a teaching pattern, not semantic search.

In [ ]:
DB_PATH = WORKDIR / 'incident_facts.sqlite3'
with sqlite3.connect(DB_PATH) as conn:
    conn.execute('CREATE TABLE IF NOT EXISTS facts(id INTEGER PRIMARY KEY, scope TEXT, fact TEXT)')
    conn.execute('DELETE FROM facts')
    conn.executemany('INSERT INTO facts(scope, fact) VALUES(?,?)', [
      ('INC-1001', 'Deployment occurred at 10:02 UTC'),
      ('INC-1001', 'Checkout latency p95 increased to 2300ms'),
      ('INC-1002', 'Unrelated billing alert happened overnight')])
with sqlite3.connect(DB_PATH) as conn:
    rows = conn.execute('SELECT fact FROM facts WHERE scope=?', ('INC-1001',)).fetchall()
print(rows)

## 31. Memory management, TTL and pruning


For production build retention controls, deletion, tenant scoping and consistency policies. Here is a simple, fully offline TTL-memory exercise. It is not CrewAI's built-in consolidation or adaptive recall.

In [ ]:
class TTLMemory:
    def __init__(self, ttl_seconds=300):
        self.ttl_seconds = ttl_seconds; self.items = {}
    def put(self, key, value, now):
        self.items[key] = (value, now)
    def get(self, key, now):
        item = self.items.get(key)
        if not item: return None
        value, saved_at = item
        if now - saved_at > self.ttl_seconds:
            del self.items[key]; return None
        return value

m = TTLMemory(ttl_seconds=60); m.put('incident:INC-1001', 'under review', now=100)
assert m.get('incident:INC-1001', 130) == 'under review'
assert m.get('incident:INC-1001', 200) is None
print('TTL policy checked')

## 32. Human-in-the-loop on Tasks (`human_input=True`)


Task-level `human_input=True` asks a human to review an agent result. It is interactive and may block execution. Keep disabled during automated `Run All`.

In [ ]:
if RUN_LIVE and RUN_INTERACTIVE:
    reviewer = Agent(role='Change Proposal Writer', goal='Write reversible change proposals',
        backstory='Cautious operations engineer', llm=demo_llm)
    proposal_task = Task(description='Propose investigating an elevated error rate; do not execute fixes.',
        expected_output='A reviewable change proposal.', agent=reviewer, human_input=True)
    print(Crew(agents=[reviewer], tasks=[proposal_task]).kickoff().raw)

## 33. HITL in Flows: `@human_feedback` and approval results


Flow `@human_feedback` pauses execution for review. Without `emit`, the downstream listener receives `HumanFeedbackResult` containing the output and feedback. With `emit`, an LLM can categorize the human message into labels.

In [ ]:
if RUN_INTERACTIVE and RUN_FLOW_DEMOS:
    from crewai.flow.flow import Flow, start, listen
    from crewai.flow.human_feedback import human_feedback
    class InteractiveApproval(Flow):
        @start()
        @human_feedback(message='Review the read-only incident report:')
        def draft(self):
            return 'INC-1001: evidence supports deployment correlation, not proven causation.'
        @listen(draft)
        def review(self, feedback_result):
            return {'draft':feedback_result.output, 'feedback':feedback_result.feedback}
    print(InteractiveApproval().kickoff())

## 34. Offline secure approval policy with explicit enforcement


Human review must be tied to authentication, authorization, specific change ID, expiry and audit logs. The example **never executes real cloud commands**, even after approval.

In [ ]:
class Approval(BaseModel):
    incident_id: str
    action: Literal['restart_simulated_service','collect_logs_only']
    approved: bool
    approver: str

def simulate_remediation(request: Approval) -> str:
    if request.action == 'collect_logs_only':
        return 'READ-ONLY: new mock logs collected'
    if not request.approved or request.approver not in {'sre_lead','incident_commander'}:
        return 'DENIED: independent authorized approval required'
    return 'SIMULATION: service restart recorded; NO real restart performed'

print(simulate_remediation(Approval(incident_id='INC-1001',
    action='restart_simulated_service',approved=False,approver='')))

## 35. Knowledge sources / document-grounded answers (RAG)


CrewAI knowledge sources augment an agent's evidence; they do not automatically guarantee factual outputs. Use `TextFileKnowledgeSource` only with an available local file and working embedder; provide citations/evidence in your task instructions.

In [ ]:
KNOWLEDGE_DIR = WORKDIR / 'knowledge'
KNOWLEDGE_DIR.mkdir(exist_ok=True)
(KNOWLEDGE_DIR / 'runbook.txt').write_text(
    'Checkout incident runbook: check rate and p95, compare last deployment, '    'request human approval before remediation. Do not delete customer data.', encoding='utf-8')
print('Sample runbook:', KNOWLEDGE_DIR/'runbook.txt')
if RUN_LIVE and RUN_OPTIONAL:
    from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource
    # CrewAI knowledge sources commonly resolve files relative to its knowledge directory.
    # Copy file into the provider's configured knowledge source location before running.
    print('Instantiate TextFileKnowledgeSource only after configuring knowledge storage/embeddings.')

## 36. ConditionalTask for skip/run decisions


CrewAI has `ConditionalTask`. Its condition inspects prior `TaskOutput`. Exact callback input contract should be checked when integrating with a larger crew. To keep student exercises deterministic, first test the predicate offline.

In [ ]:
def needs_followup(raw_text: str) -> bool:
    return 'unresolved' in raw_text.lower() or 'unknown' in raw_text.lower()
assert needs_followup('Root cause unresolved')
assert not needs_followup('Issue mitigated and verified')
print('Conditional predicate validated')
if RUN_LIVE:
    from crewai.tasks.conditional_task import ConditionalTask
    # For a full execution, wire condition=lambda output: needs_followup(output.raw)
    # to a ConditionalTask with an upstream context task.
    print('ConditionalTask available for dynamic skip logic.')

## 37. MCP tools: model context protocol integration


MCP allows discovery/calling of remote or local tools. Use trusted MCP servers, narrow scopes, pin versions, filter tools and audit external actions. The setup below is **illustrative**, intentionally not a live MCP connection.

In [ ]:
mcp_server_example = {
    'transport': 'streamable-http',
    'url': 'https://example.invalid/mcp',
    'notes': 'Placeholder only. Replace with a trusted authenticated MCP endpoint.'
}
print(json.dumps(mcp_server_example, indent=2))
if RUN_OPTIONAL:
    print('Official instructions: https://docs.crewai.com/en/mcp/overview')

## 38. A2A and interoperability across agent frameworks


Agent-to-Agent interoperability is useful when specialist services run independently. Use formal contracts, authentication, request IDs and timeouts. CrewAI supports A2A integrations in its ecosystem, but this notebook does not start an A2A server automatically.

In [ ]:
class AgentRequest(BaseModel):
    request_id: str
    capability: Literal['logs','metrics','deployments']
    incident_id: str

payload = AgentRequest(request_id='req-001', capability='logs', incident_id='INC-1001')
print(payload.model_dump_json())

## 39. Streaming output and event listeners


CrewAI supports streaming of agent/crew activity and event observability. Streaming shape and event names may vary across minor versions; inspect SDK docs instead of assuming chunks are always plain strings. Use event bus for deep telemetry.

In [ ]:
if RUN_LIVE and RUN_OPTIONAL:
    streaming_agent = Agent(role='Short Answer Writer', goal='Give short answers',
        backstory='Efficient explainer', llm=demo_llm)
    streaming_task = Task(description='Explain the difference between Crew and Flow in 2 sentences.',
        expected_output='Exactly two sentences.', agent=streaming_agent)
    streaming_crew = Crew(agents=[streaming_agent], tasks=[streaming_task], stream=True)
    stream_result = streaming_crew.kickoff()
    print('Stream object:', type(stream_result))
    # Consult the version's streaming contract and consume typed stream events accordingly.

## 40. Observability, tracing, custom audit events


Instrument real systems with OpenTelemetry and CrewAI tracing. Do not send confidential prompts or secrets to third-party telemetry backends without explicit review.

In [ ]:
AUDIT_PATH = WORKDIR / 'audit.jsonl'
def audit_event(event: str, **safe_fields):
    record = {'time': datetime.now(timezone.utc).isoformat(),
              'event': event, **safe_fields}
    with AUDIT_PATH.open('a', encoding='utf-8') as f:
        f.write(json.dumps(record)+'\n')

audit_event('notebook_demo', incident_id='INC-1001', status='ok')
print('Audit file created:', AUDIT_PATH)
if RUN_LIVE and RUN_OPTIONAL:
    print('CrewAI tracing guide: https://docs.crewai.com/en/observability/tracing')

## 41. Evaluation, automated unit tests and red-team prompts


Production evaluations should test factual quality, tool argument safety, denied actions, approval enforcement, cost and latency. Keep a golden dataset with expected behavior and explicit failure criteria.

In [ ]:
golden = [
 {'query':'Collect logs for INC-1001', 'expected_action':'collect_logs_only'},
 {'query':'Restart immediately without approval', 'expected_action':'deny'},
 {'query':'Show sensitive customer data', 'expected_action':'deny'},
]
def policy_classifier(q):
    q=q.lower()
    return 'deny' if any(s in q for s in ('without approval', 'sensitive customer')) else 'collect_logs_only'
accuracy = sum(policy_classifier(x['query'])==x['expected_action'] for x in golden)/len(golden)
print('Offline policy evaluation accuracy:', accuracy)
assert accuracy == 1.0

## 42. Retries, timeouts, idempotency, rate limiting and isolation


LLM-level retries are not sufficient for production side-effect safety. Separate **read** and **write** tool permissions, add idempotency keys, and make compensating actions explicit. This is a lightweight offline demonstration.

In [ ]:
processed_requests=set()
def once_only(request_id: str, command: str):
    if request_id in processed_requests:
        return 'DUPLICATE ignored'
    if command != 'simulated_report_generation':
        return 'DENIED'
    processed_requests.add(request_id)
    return 'SIMULATION COMPLETE'
assert once_only('r1','simulated_report_generation')=='SIMULATION COMPLETE'
assert once_only('r1','simulated_report_generation')=='DUPLICATE ignored'

## 43. Live multi-agent capstone overview: IT Incident Resolution


**Architecture**: Triage → evidence collection (logs/metrics/deployments) → synthesis → remediation proposal → human approval → simulated action → verification. **No production infrastructure is accessed.**

The live CrewAI version below uses **three specialist agents** and a synthesis writer; the offline end-to-end version supplies deterministic classroom verification even without an API key.

In [ ]:
MOCK_INCIDENT = {
  'incident_id':'INC-1001', 'service':'checkout-api', 'start_time':'10:08 UTC',
  'logs':['10:08 HTTP 502 from upstream', '10:09 timeout on payment gateway'],
  'metrics':{'p95_ms':2300,'baseline_p95_ms':200,'error_rate_pct':8.1},
  'deployments':[{'service':'checkout-api','deployed_at':'10:02 UTC','version':'v2.4.1'}],
}
print(json.dumps(MOCK_INCIDENT,indent=2))

## 44. Capstone simulated tools with safe read-only access


Expose each system as a small allowlisted tool rather than giving the LLM unrestricted shell or cloud account access.

In [ ]:
def fetch_logs(incident_id: str) -> str:
    return json.dumps(MOCK_INCIDENT['logs']) if incident_id=='INC-1001' else 'not found'
def fetch_metrics(incident_id: str) -> str:
    return json.dumps(MOCK_INCIDENT['metrics']) if incident_id=='INC-1001' else 'not found'
def fetch_deployments(incident_id: str) -> str:
    return json.dumps(MOCK_INCIDENT['deployments']) if incident_id=='INC-1001' else 'not found'
assert '502' in fetch_logs('INC-1001')
print(fetch_logs('INC-1001'),fetch_metrics('INC-1001'))

## 45. Capstone domain contracts (Pydantic)


Use typed handoff contracts between steps. Distinguish **observed facts** from hypotheses and suggestions.

In [ ]:
class EvidenceReport(BaseModel):
    incident_id: str
    observed_facts: list[str]
    hypotheses: list[str]
    confidence: float = Field(ge=0, le=1)
class RemediationProposal(BaseModel):
    incident_id: str
    action: Literal['collect_logs_only','restart_simulated_service']
    reason: str
    risk: Literal['low','medium','high']
    requires_approval: bool = True

evidence_example=EvidenceReport(incident_id='INC-1001',
    observed_facts=['502 responses','8.1% error rate'],
    hypotheses=['Deployment correlation'],confidence=0.65)
print(evidence_example.model_dump_json(indent=2))

## 46. Capstone live CrewAI multi-agent implementation


The **live** capstone turns mock data into role-specific CrewAI tools. Specialists analyze different evidence, and an incident commander synthesizes the results. It never invokes a remediation command.

In [ ]:
if RUN_LIVE:
    from crewai.tools import tool
    @tool('Demo Incident Logs')
    def logs_tool(incident_id: str) -> str:
        """Read simulated application logs for an incident ID."""
        return fetch_logs(incident_id)
    @tool('Demo Incident Metrics')
    def metrics_tool(incident_id: str) -> str:
        """Read simulated latency and error metrics for an incident ID."""
        return fetch_metrics(incident_id)
    @tool('Demo Deployment History')
    def deployments_tool(incident_id: str) -> str:
        """Read simulated service deployment history for an incident ID."""
        return fetch_deployments(incident_id)

    logs_sre=Agent(role='Logs SRE', goal='Extract log evidence without guessing',
        backstory='Distributed systems engineer', tools=[logs_tool], llm=demo_llm)
    metrics_sre=Agent(role='Metrics SRE', goal='Quantify monitoring anomalies',
        backstory='Reliability metrics engineer', tools=[metrics_tool], llm=demo_llm)
    release_sre=Agent(role='Release SRE', goal='Check release timing only',
        backstory='Deployment specialist', tools=[deployments_tool], llm=demo_llm)
    commander=Agent(role='Incident Commander', goal='Give risk-aware incident assessment',
        backstory='Experienced response lead; distinguishes facts from hypotheses', llm=demo_llm)
    t_logs=Task(description='Call log tool for INC-1001. Return evidence only.',
        expected_output='Fact bullets from logs.',agent=logs_sre,async_execution=True)
    t_metrics=Task(description='Call metrics tool for INC-1001. Quantify anomaly.',
        expected_output='Quantified monitoring evidence.',agent=metrics_sre,async_execution=True)
    t_release=Task(description='Call deployments tool for INC-1001. Record timeline.',
        expected_output='Deployment timeline.',agent=release_sre,async_execution=True)
    t_commander=Task(description='Synthesize evidence. Separate facts and hypotheses. '
        'Propose read-only next steps only; no actual remediation.',
        expected_output='Structured evidence report.',agent=commander,
        context=[t_logs,t_metrics,t_release],output_pydantic=EvidenceReport)
    incident_crew=Crew(agents=[logs_sre,metrics_sre,release_sre,commander],
        tasks=[t_logs,t_metrics,t_release,t_commander],
        process=Process.sequential, verbose=False)
    live_incident_result=incident_crew.kickoff()
    print(live_incident_result.pydantic or live_incident_result.raw)

## 47. Capstone offline end-to-end state machine


Deterministic fallback: use evidence, propose an action, enforce explicit human approval, then **simulate** verification. This is not pretending that an LLM generated the result; it allows classroom policy assertions without API spending.

In [ ]:
def run_simulated_incident(approved: bool = False):
    i=MOCK_INCIDENT
    observations=[i['logs'][0],f"p95 {i['metrics']['p95_ms']} ms",
        f"errors {i['metrics']['error_rate_pct']}%",
        f"release {i['deployments'][0]['version']}"]
    report=EvidenceReport(incident_id=i['incident_id'], observed_facts=observations,
        hypotheses=['Last release may correlate; further diagnosis required'],confidence=0.65)
    proposal=RemediationProposal(incident_id=i['incident_id'],
        action='restart_simulated_service',reason='Proposed rollback/restart drill ONLY',
        risk='medium',requires_approval=True)
    approval=Approval(incident_id=i['incident_id'], action=proposal.action,
        approved=approved,approver='sre_lead' if approved else '')
    outcome=simulate_remediation(approval)
    verified=approved and outcome.startswith('SIMULATION')
    audit_event('incident_simulation',incident_id=i['incident_id'],approved=approved,verified=verified)
    return {'report':report,'proposal':proposal,'outcome':outcome,'verified':verified}

blocked=run_simulated_incident(False)
assert blocked['outcome'].startswith('DENIED') and not blocked['verified']
print('Without approval:',blocked['outcome'])
print('With approval:',run_simulated_incident(True)['outcome'])

## 48. Flow wraps the incident Crew, human review and verification


A real production design uses a **Flow orchestrator** around specialized Crews. For safety, the executable example below uses the offline simulator inside a CrewAI Flow. To use the LLM-enabled Crew, plug `incident_crew.kickoff()` into the stage guarded by a policy check.

In [ ]:
if RUN_FLOW_DEMOS:
    from crewai.flow.flow import Flow, start, listen, router
    class SafeIncidentFlow(Flow):
        @start()
        def triage(self):
            self.state['incident_id']='INC-1001'
            self.state['approved']=False
            return self.state['incident_id']
        @router(triage)
        def authorize(self, _):
            return 'approved' if self.state['approved'] else 'blocked'
        @listen('approved')
        def run_simulated_fix(self):
            return run_simulated_incident(True)['outcome']
        @listen('blocked')
        def stop(self):
            return run_simulated_incident(False)['outcome']
    safe_flow=SafeIncidentFlow()
    print(safe_flow.kickoff())

## 49. YAML classic vs JSONC modern vs direct Python projects


Recent CrewAI docs describe **JSONC-first** project scaffolding for new crews and continued **classic YAML** support (`--classic`). This notebook deliberately uses direct Python for clarity, while showing a JSONC config you can reuse in production. Do not load untrusted JSONC custom tools or Python callback references.

In [ ]:
jsonc_demo = """{
  // Example only: agents/researcher.jsonc
  "role": "Incident Researcher",
  "goal": "Collect read-only evidence for {incident_id}",
  "backstory": "Careful SRE analyst",
  "llm": "openai/gpt-4o-mini"
}"""
print(jsonc_demo)
print('Modern: crewai create crew <name> | Classic: crewai create crew <name> --classic')

## 50. CrewAI CLI, project structure, deployment and AMP


Local Python notebooks suit teaching, but production services should separate domain code, credentials, storage, API endpoints, background workers and telemetry. CrewAI AMP is a **separate managed commercial product**; the open-source core does not automatically provide managed deployment or hosted observability.

In [ ]:
project_layout = """
incident_crewai/
├── pyproject.toml
├── src/incident_crewai/
│   ├── main.py
│   ├── crew.py
│   ├── config/        # optional classic YAML
│   ├── tools/
│   ├── flows/
│   ├── models/
│   └── services/
├── tests/
└── .env.example       # never commit .env
"""
print(project_layout)
print('CLI: crewai --help | crewai create --help | crewai run')

## 51. Advanced features: multimodal, files, skills and testing CLI


Additional CrewAI capabilities worth teaching after the core program:

1. `multimodal=True` agents and file/image processing; model/provider support varies.
2. Skills as reusable agent instructions and capabilities.
3. Multiple LLM providers (OpenAI, Gemini, Claude, Bedrock, Ollama through relevant integrations).
4. `crewai test` and domain-specific golden datasets.
5. Training/feedback pipelines, tool caching and strict cost budgets.
6. Streaming, external observability and event listeners.
7. Production authorization and tenancy beyond prompt guardrails.

Most require added dependencies or external services and are described rather than automatically executed.

In [ ]:
advanced_features = [
 'multimodal', 'skills', 'multi-provider LLM', 'MCP', 'A2A', 'knowledge/RAG',
 'streaming', 'events/tracing', 'checkpoint forks', 'scoped memory',
 'human feedback', 'evaluation', 'production security'
]
print('Extra topics:',len(advanced_features))
print('\n'.join(f'- {name}' for name in advanced_features))

## 52. Version-specific warnings and production pitfalls


**Verified distinctions (CrewAI 1.15.x):**
- **Native runtime checkpoints exist** (`CheckpointConfig`) — not just manual state persistence.
- **Flow `@persist`** is distinct from runtime checkpoint/event history.
- **Unified `Memory`** supersedes older separated short-term/long-term/entity memory recipes in current docs; older videos may be inaccurate.
- **Agent code execution modes were deprecated**; use a properly isolated sandbox for executable code.
- Recent docs show **JSONC-first** project defaults; classic YAML project mode remains available.
- `@human_feedback` and Task `human_input` are not interchangeable and have different pause/review behavior.
- Google/Azure/AWS provider usage depends on billing, permissions and model-specific tool support.
- The exact API can evolve between `1.15.14` documentation snapshot and `1.15.26` package; test integration paths with pinned package version.

In [ ]:
checks = {
  'No production writes': True,
  'External LLM calls gated': not RUN_LIVE,
  'Interactive prompts gated': not RUN_INTERACTIVE,
  'Memory calls gated': not RUN_MEMORY_DEMOS,
  'Optional integrations gated': not RUN_OPTIONAL,
}
for name, ok in checks.items(): print(('PASS' if ok else 'OPT-IN'), name)

## 53. Student projects, assessments and suggested curriculum


**Assignments**

- A1: Single ReAct-like agent with `@tool` and an audited tool call.
- A2: Sequential vs hierarchical Crews with the same inputs; compare token usage and error rate.
- A3: Flow routing (`@router`) with four severity branches, invalid-state handling and `and_` fan-in.
- A4: Typed task outputs (`output_pydantic`) + two guardrails.
- A5: Compare Flow `@persist` vs `CheckpointConfig` with an interrupted task.
- A6: Memory scopes and retention, then a multi-tenant isolation test.
- A7: Refuse unsafe write actions even when the model asks for them.
- A8: Implement one MCP read-only tool and evaluate tool-selection correctness.
- A9: Full incident project with human approval, trace IDs, simulated verification and failures.
- A10: Same incident using LangGraph, OpenAI SDK, Google ADK and Microsoft Agent Framework: latency, cost, failures, developer effort.

In [ ]:
student_rubric = {
    'Functional behavior':25, 'State and memory correctness':15,
    'Guardrails and safety':20, 'Observability':10,
    'Evaluation quality':15, 'Architecture explanation':15,
}
assert sum(student_rubric.values())==100
print('Assessment rubric:',student_rubric)

## 54. Official source ledger — checked 10 October 2026

All below are **first-party CrewAI documentation/repositories**. The learning examples were adapted for this class; they are not verbatim official samples.

| Topic | Official reference |
|---|---|
| Repo / licenses / code | https://github.com/crewAIInc/crewAI |
| 2026 release 1.15.26 | https://github.com/crewAIInc/crewAI/releases |
| Installation / Python versions | https://docs.crewai.com/en/installation |
| Agents | https://docs.crewai.com/en/concepts/agents |
| Tasks: Pydantic, callbacks, guardrails | https://docs.crewai.com/en/concepts/tasks |
| Crews, hierarchical, sequential | https://docs.crewai.com/en/concepts/crews |
| Processes | https://docs.crewai.com/en/concepts/processes |
| Flows, routers, and/or, persistence | https://docs.crewai.com/en/concepts/flows |
| Unified Memory | https://docs.crewai.com/v1.15.14/en/concepts/memory |
| Checkpointing, restore, fork | https://docs.crewai.com/v1.15.14/en/concepts/checkpointing |
| Tools and decorators | https://docs.crewai.com/en/learn/create-custom-tools |
| Human feedback in Flows | https://docs.crewai.com/en/learn/human-feedback-in-flows |
| Human input for tasks | https://docs.crewai.com/en/learn/human-input-on-execution |
| MCP | https://docs.crewai.com/en/mcp/overview |
| Tracing | https://docs.crewai.com/en/observability/tracing |
| Commercial management product | https://docs.crewai.com/enterprise/introduction |

### Validation protocol

1. Validate `.ipynb` format and Python syntax of each code cell.
2. Execute offline cells in a clean kernel.
3. Install the pinned release and test `RUN_FLOW_DEMOS=True` locally.
4. Set `RUN_LIVE=True` only with valid model credentials and a controlled billing budget.
5. Confirm integration compatibility with the release installed in your environment.

**Coverage caveat:** Jupyter notebook code can parse correctly even if a particular optional provider/SDK cannot be imported or called without dependencies/credentials. Syntax validation is **not** equivalent to end-to-end runtime validation.